# 1. Profile the bag to get metadata + topics

In [1]:
import sys
from pathlib import Path

demo_dir = Path.cwd()
repo_root = demo_dir.parent if demo_dir.name == "demo" else demo_dir
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from hephaes import Profiler

bag_files = [
    str(Path("input") / "ros2.mcap"),
]
topics = []

if not bag_files:
    print("No bag files configured.")
else:
    prof = Profiler(bag_files, max_workers=1)
    metadata_list = prof.profile()
    for meta in metadata_list:
        print(f"File: {meta.file_path}")
        print(f"Path: {meta.path}")
        print(f"ROS version: {meta.ros_version}")
        print(f"Storage format: {meta.storage_format}")
        print(f"File size (bytes): {meta.file_size_bytes}")
        print(f"Start: {meta.start_time_iso} End: {meta.end_time_iso} Duration(s): {meta.duration_seconds}")
        print(f"Message count: {meta.message_count}")
        print("Topics:")
        topics = meta.topics
        for t in meta.topics:
            print(f"  - {t.name} ({t.message_type}): {t.message_count} messages, {t.rate_hz} Hz")
        print("-" * 40)


File: input/ros2.mcap
Path: input/ros2.mcap
ROS version: ROS2
Storage format: mcap
File size (bytes): 9566091
Start: 2023-10-06T07:06:59.485776Z End: 2023-10-06T07:10:17.857374Z Duration(s): 198.371597915
Message count: 44416
Topics:
  - /nissan/gps/duro/current_pose (geometry_msgs/msg/PoseStamped): 3967 messages, 20.01 Hz
  - /nissan/gps/duro/imu (sensor_msgs/msg/Imu): 19662 messages, 99.12 Hz
  - /nissan/gps/duro/mag (sensor_msgs/msg/MagneticField): 4916 messages, 24.78 Hz
  - /nissan/gps/duro/status_string (std_msgs/msg/String): 3967 messages, 20.01 Hz
  - /nissan/vehicle_speed (std_msgs/msg/Float32): 5952 messages, 30.01 Hz
  - /nissan/vehicle_steering (std_msgs/msg/Float32): 5952 messages, 30.01 Hz
----------------------------------------


# 2. Create a MappingTemplate

In [2]:
from hephaes.mappers import build_mapping_template

mapping = build_mapping_template(topics)

# 3. Use Converter to write parquet output


In [3]:
from pathlib import Path
from hephaes import Converter, ParquetOutputConfig, ResampleConfig, TFRecordOutputConfig

output_root = Path("./output")
parquet_output_dir = output_root / "parquet"
tfrecord_output_dir = output_root / "tfrecord"

# Default: no resampling (union timestamps, null when a column has no value).
# parquet_files = Converter(bag_files, mapping, parquet_output_dir).convert()

# Optional: regular-grid downsample or interpolate.
resample_cfg = ResampleConfig(freq_hz=10.0, method="downsample")
parquet_files = Converter(
    bag_files,
    mapping,
    parquet_output_dir,
    output=ParquetOutputConfig(),
    resample=resample_cfg,
).convert()

parquet_files


[PosixPath('output/parquet/episode_0001.parquet')]

# 4. Stream the first few rows of the parquet output


In [4]:
from hephaes import stream_wide_parquet_rows

output_parquet = parquet_files[0]
n_rows = 5

for i, row in enumerate(stream_wide_parquet_rows(output_parquet)):
    print(row)
    if i + 1 >= n_rows:
        break


{'timestamp_ns': 1696576019485776332, 'nissan_gps_duro_current_pose': '{"__bytes__":true,"encoding":"base64","value":"AAEAABOyH2XmTuchBAAAAG1hcAC+H1pn4eeVv9Wh5gZUV50/AAAAAAAAAACAVOBIctYvv+DrNfHB/hq/Lhxhy16oSj8BNKg7///vPw=="}', 'nissan_gps_duro_imu': '{"__bytes__":true,"encoding":"base64","value":"AAEAABOyH2VbStMiBQAAAGR1cm8AAAAAAAAAALDY9Snkj1m/mkZpbUQ7fT+gKz0lC9jtP5WAYzkDF9c/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAW839O0MmOj+BY9gEUxqLPyFJqCseVHw/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAiPTb14FrvT/3deCcEb6pv8SxLm6e1iPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"}', 'nissan_gps_duro_mag': '{"__bytes__":true,"encoding":"base64","value":"AAEAABOyH2WsEPkhBQAAAGR1cm8AAAAAAAAAAKIm+nyUEfc+3gAz38FP/L4/q8yU1t8CvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA=="}', 'nissan_gps_duro_stat

# 5. Convert the same bag to TFRecord


In [5]:
tfrecord_files = Converter(
    bag_files,
    mapping,
    tfrecord_output_dir,
    output=TFRecordOutputConfig(),
    resample=resample_cfg,
).convert()

tfrecord_files


[PosixPath('output/tfrecord/episode_0001.tfrecord')]

# 6. Stream the first few rows of the TFRecord output


In [6]:
from hephaes import stream_tfrecord_rows

output_tfrecord = tfrecord_files[0]

for i, row in enumerate(stream_tfrecord_rows(output_tfrecord)):
    print(row)
    if i + 1 >= n_rows:
        break


{'timestamp_ns': 1696576019485776332, 'nissan_gps_duro_current_pose__present': 1, 'nissan_gps_duro_current_pose__header__stamp__sec': 1696576019, 'nissan_gps_duro_current_pose__header__stamp__nanosec': 568807142, 'nissan_gps_duro_current_pose__header__frame_id': b'map', 'nissan_gps_duro_current_pose__pose__position__x': -0.02139236591756344, 'nissan_gps_duro_current_pose__pose__position__y': 0.028653442859649658, 'nissan_gps_duro_current_pose__pose__position__z': 0.0, 'nissan_gps_duro_current_pose__pose__orientation__x': -0.00024290222791023552, 'nissan_gps_duro_current_pose__pose__orientation__y': -0.00010297831613570452, 'nissan_gps_duro_current_pose__pose__orientation__z': 0.0008135283133015037, 'nissan_gps_duro_current_pose__pose__orientation__w': 0.9999996423721313, 'nissan_gps_duro_imu__present': 1, 'nissan_gps_duro_imu__header__stamp__sec': 1696576019, 'nissan_gps_duro_imu__header__stamp__nanosec': 584272475, 'nissan_gps_duro_imu__header__frame_id': b'duro', 'nissan_gps_duro_imu